In [ ]:
# Read files..
import pandas as pd
from pathlib import Path

scored_validation = pd.read_csv("../../Data/processed/scored_validation_transactions.csv")
model_comparison = pd.read_csv("../../Data/processed/model_comparison.csv")
threshold_comparison = pd.read_csv("../../Data/processed/threshold_comparison.csv")
tuned_feature_importance = pd.read_csv("../../Data/processed/tuned_feature_importance.csv")

# Verify the transaction export
#assert scored_validation.shape == (118108, 41)
#assert scored_validation["TransactionID"].is_unique
#assert scored_validation["ActualFraud"].sum() == 4064
#assert scored_validation["PredictedFraud"].sum() == 3424


(118108, 41)


In [ ]:
# Transform the data to the SQL DB 
# %pip install sqlalchemy pyodbc
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# List installed database drivers
#print(pyodbc.drivers())

# Connect to the local SQL Server using Windows Authentication
connection_url = URL.create("mssql+pyodbc",
    query={
        "odbc_connect": (
            "DRIVER={ODBC Driver 18 for SQL Server};"
            "SERVER=localhost;"
            "DATABASE=FraudDetection;"
            "Trusted_Connection=yes;"
            "Encrypt=yes;"
            "TrustServerCertificate=yes;"
        )
    }
)

engine = create_engine(connection_url,fast_executemany=True)
with engine.connect() as connection:database_name = connection.execute(text("SELECT DB_NAME()")).scalar_one()

print("DB name:", database_name)


Connected to: FraudDetection


In [ ]:
# Inspect column types & missing values, so that I can create SQL tables... 
column_profile = pd.DataFrame({
    "DataType": scored_validation.dtypes.astype(str),
    "MissingCount": scored_validation.isna().sum()
})

# Measure text lengths for SQL column sizing
column_profile["MaxTextLength"] = [
    scored_validation[column].dropna().astype(str).str.len().max()
    if pd.api.types.is_string_dtype(scored_validation[column])
    or scored_validation[column].dtype == "object"
    else None
    for column in scored_validation.columns
]

display(column_profile)

,DataType,MissingCount,MaxTextLength
TransactionID,int64,0,NaN
TransactionDT,int64,0,NaN
TranHour,int64,0,NaN
TransactionAmt,float64,0,NaN
ProductCD,str,0,1.0
card1,int64,0,NaN
card2,float64,1908,NaN
card3,float64,742,NaN
card4,str,744,16.0
card5,float64,1106,NaN


In [12]:
from sqlalchemy import (MetaData, Table, Column, Integer, Float, Numeric, Unicode)
from sqlalchemy.dialects.mssql import BIT

# Match each DataFrame to its SQL table
report_tables = {
    "ScoredValidation": scored_validation,
    "ModelComparison": model_comparison,
    "ThresholdComparison": threshold_comparison,
    "FeatureImportance": tuned_feature_importance
}

primary_keys = {
    "ScoredValidation": "TransactionID",
    "ModelComparison": "Model",
    "ThresholdComparison": "Threshold",
    "FeatureImportance": "Feature"
}

# Define explicit types for transaction columns
transaction_types = {
    "TransactionID": Integer(),
    "TransactionDT": Integer(),
    "TranHour": Integer(),
    "TransactionAmt": Numeric(18, 3),
    "card1": Integer(),
    "HasIdentity": BIT(),
    "ActualFraud": BIT(),
    "PredictedFraud": BIT(),
    "ProductCD": Unicode(5),
    "card4": Unicode(30),
    "card6": Unicode(20),
    "DeviceType": Unicode(20),
    "DeviceInfo": Unicode(100),
    "P_emaildomain": Unicode(100),
    "R_emaildomain": Unicode(100),
    "id_31": Unicode(100),
    "ModelVersion": Unicode(50)
}

metadata = MetaData(schema="dbo")

for table_name, dataframe in report_tables.items():
    columns = []

    # Validate the primary key before loading
    key = primary_keys[table_name]
    assert dataframe[key].notna().all()
    assert dataframe[key].is_unique

    for column_name in dataframe.columns:
        if (
            table_name == "ScoredValidation"
            and column_name in transaction_types
        ):
            sql_type = transaction_types[column_name]

        elif pd.api.types.is_integer_dtype(dataframe[column_name]):
            sql_type = Integer()

        elif pd.api.types.is_numeric_dtype(dataframe[column_name]):
            sql_type = Float(53)

        else:
            sql_type = Unicode(200)

        columns.append(
            Column(
                column_name,
                sql_type,
                primary_key=(column_name == key),
                nullable=(column_name != key),
                autoincrement=False

            )
        )

    Table(table_name, metadata, *columns)
print("Table definitions ready.")

Table definitions ready.


In [ ]:
# Create and load all four tables in one transaction
with engine.begin() as connection:

    assert connection.execute(
        text("SELECT DB_NAME()")
    ).scalar_one() == "FraudDetection"

    metadata.create_all(connection)

    # Stop if any target table already contains data
    for table_name in report_tables:
        existing_rows = connection.execute(
            text(f"SELECT COUNT_BIG(*) FROM dbo.[{table_name}]")
        ).scalar_one()

        if existing_rows > 0:
            raise ValueError( f"{table_name} already contains data. "
                "Loading stopped to prevent duplicates.")

    # Load data in batches and verify row counts
    for table_name, dataframe in report_tables.items():
        dataframe.to_sql(
            name=table_name,
            con=connection,
            schema="dbo",
            if_exists="append",
            index=False,
            chunksize=1000
        )

        loaded_rows = connection.execute(text(f"SELECT COUNT_BIG(*) FROM dbo.[{table_name}]")).scalar_one()

        #assert loaded_rows == len(dataframe)
        #print(f"{table_name}: {loaded_rows:,} rows verified")

    # Verify actual fraud and alert totals
    totals = connection.execute(text("""
        SELECT
            SUM(CAST(ActualFraud AS INT)),
            SUM(CAST(PredictedFraud AS INT))
        FROM dbo.ScoredValidation
    """)).one()



print("All four tables loaded successfully.")

ScoredValidation: 118,108 rows verified
ModelComparison: 5 rows verified
ThresholdComparison: 5 rows verified
FeatureImportance: 433 rows verified
All four tables loaded successfully.
